Phase C: Real-Time Inference & Deployment Pipeline

Phase C represents the Deployment stage of the project. The objective is to integrate the static weights learned in Phase B into a live computer vision pipeline. This system utilizes a Cascade-CNN Hybrid approach to track gaze in real-time, implementing a calibration layer to ensure the system is person-agnostic and robust to environmental noise.

In [3]:
import cv2
import numpy as np
import tensorflow as tf
from collections import deque

In [ ]:


model = tf.keras.models.load_model('proctor_gaze_model.h5', compile=False)

eye_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_eye.xml')
cap = cv2.VideoCapture(0)

SENSITIVITY = 0.02  
CALIB_FRAMES_REQUIRED = 50
status_buffer = deque(maxlen=15) 

calibrated = False
base_x, base_y = 0, 0
calib_count = 0

print("--- DIAGNOSTIC PROCTOR ACTIVE ---")
print("STEP 1: Stare at the center for calibration...")

while True:
    ret, frame = cap.read()
    if not ret: break
    frame = cv2.flip(frame, 1)
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    
    eyes = eye_cascade.detectMultiScale(gray, 1.3, 5)
    
    current_status = "SAFE"
    color = (0, 255, 0)

    for (x, y, w, h) in eyes:
        
        roi = gray[y:y+h, x:x+w]
        roi = cv2.equalizeHist(roi) 
        roi = cv2.resize(roi, (64, 64))
        roi = roi.astype('float32') / 255.0
        roi_input = np.reshape(roi, (1, 64, 64, 1))
        
        preds = model.predict(roi_input, verbose=0)[0]
        curr_x, curr_y = preds[0], preds[1]

        if not calibrated:
            base_x += curr_x
            base_y += curr_y
            calib_count += 1
            cv2.putText(frame, f"CALIBRATING: {calib_count}%", (x, y-10), 1, 1, (0, 255, 255), 2)
            
            if calib_count >= CALIB_FRAMES_REQUIRED:
                base_x /= CALIB_FRAMES_REQUIRED
                base_y /= CALIB_FRAMES_REQUIRED
                calibrated = True
                print(f"\nCalibration Complete! Base: [{base_x:.3f}, {base_y:.3f}]")
        else:
            
            diff_x = curr_x - base_x
            diff_y = curr_y - base_y
            
            print(f"RAW: {curr_x:.3f} | DIFF: {abs(diff_x):.3f}", end='\r')

            cx, cy = x + w//2, y + h//2
           
            target_x = int(cx + diff_x * 400)
            target_y = int(cy + diff_y * 400)
            cv2.line(frame, (cx, cy), (target_x, target_y), (255, 0, 0), 2)

        
            if abs(diff_x) > SENSITIVITY or abs(diff_y) > SENSITIVITY:
                status_buffer.append(1)
            else:
                status_buffer.append(0)

            if sum(status_buffer) > 10: 
                current_status = "VIOLATION DETECTED"
                color = (0, 0, 255)

        cv2.rectangle(frame, (x, y), (x+w, y+h), color, 2)

    cv2.putText(frame, f"STATUS: {current_status}", (20, 50), 
                cv2.FONT_HERSHEY_SIMPLEX, 1, color, 3)
    cv2.imshow('AI Proctor Resume Demo', frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

--- DIAGNOSTIC PROCTOR ACTIVE ---
STEP 1: Stare at the center for calibration...

Calibration Complete! Base: [0.176, -0.461]


Final Project Conclusion

This VisionProctor system successfully bridges the gap between deep learning research and practical application. By combining a lightweight CNN architecture with robust real-time logic, it achieves high-fidelity gaze tracking capable of running on standard consumer hardware (8GB RAM) without the need for specialized eye-tracking sensors.